# Which Treatments Have the Best Outcomes in Long COVID?

## Abstract

This analysis examines 6,815 treatment reports from 1,121 unique users across the r/covidlonghaulers community to identify which treatments are most frequently reported as beneficial for Long COVID. Using sentiment-scored self-reports aggregated at the user level, we apply binomial testing and 95% confidence intervals to distinguish treatments with statistically significant positive outcomes from those reflecting noise or reporting bias. Top-performing treatments include low-dose naltrexone (LDN), nattokinase, and several antihistamine-class drugs, each showing significant positive sentiment ratios across large user samples. All findings are drawn from patient-reported outcomes on social media and should not replace clinical guidance; reporting bias, selection effects, survivorship bias, and placebo response likely influence the results.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from scipy import stats
from IPython.display import display, HTML

DB_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "patientpunk.db")
if not os.path.exists(DB_PATH):
    DB_PATH = "patientpunk.db"

# Sentiment mapping: positive=1, mixed=0.5, neutral=0, negative=-1
SENTIMENT_MAP = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}

# Outcome thresholds
POS_THRESHOLD = 0.7   # mean sentiment > 0.7 => positive outcome
NEG_THRESHOLD = -0.3  # mean sentiment < -0.3 => negative outcome

# Generic treatment names to filter out
GENERICS = {"supplements", "supplement", "medication", "medications",
            "treatment", "treatments", "therapy", "therapies",
            "vitamin", "vitamins", "drug", "drugs"}

# Color palette
C_POS = "#2ecc71"
C_MIX = "#f39c12"
C_NEUT = "#95a5a6"
C_NEG = "#e74c3c"
TIER_COLORS = {"Strong": "#27ae60", "Moderate": "#f39c12", "Preliminary": "#3498db"}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "figure.dpi": 120,
})

conn = sqlite3.connect(DB_PATH)
display(HTML("<b>Setup complete.</b> Connected to database."))

## 2. Data Exploration

In [ ]:
# --- Date range ---
date_range = pd.read_sql("SELECT MIN(post_date) AS mn, MAX(post_date) AS mx FROM posts", conn)
d_min = pd.to_datetime(date_range["mn"].iloc[0], unit="s")
d_max = pd.to_datetime(date_range["mx"].iloc[0], unit="s")
n_months = max(1, round((d_max - d_min).days / 30.44))

display(HTML(
    f"<h4>Data covers: {d_min.strftime('%Y-%m-%d')} to {d_max.strftime('%Y-%m-%d')} "
    f"({n_months} month{'s' if n_months != 1 else ''})</h4>"
))

# --- Table counts ---
table_names = ["users", "posts", "treatment_reports", "treatment", "conditions"]
counts = {}
for tbl in table_names:
    counts[tbl] = pd.read_sql(f"SELECT COUNT(*) AS n FROM {tbl}", conn)["n"].iloc[0]
counts["unique_reporters"] = pd.read_sql(
    "SELECT COUNT(DISTINCT user_id) AS n FROM treatment_reports", conn
)["n"].iloc[0]

display(HTML(
    "<table style='border-collapse:collapse'>"
    "<tr style='background:#f0f0f0'><th style='padding:6px 16px;text-align:left'>Table</th>"
    "<th style='padding:6px 16px;text-align:right'>Count</th></tr>"
    + "".join(
        f"<tr><td style='padding:4px 16px'>{k}</td>"
        f"<td style='padding:4px 16px;text-align:right'>{v:,}</td></tr>"
        for k, v in counts.items()
    )
    + "</table>"
))

In [ ]:
# --- Overall sentiment distribution chart ---
sent_df = pd.read_sql(
    "SELECT sentiment, COUNT(*) AS n FROM treatment_reports GROUP BY sentiment", conn
)
sent_order = ["positive", "mixed", "neutral", "negative"]
sent_colors_map = {"positive": C_POS, "mixed": C_MIX, "neutral": C_NEUT, "negative": C_NEG}
sent_df["sentiment"] = pd.Categorical(sent_df["sentiment"], categories=sent_order, ordered=True)
sent_df = sent_df.sort_values("sentiment")

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(
    sent_df["sentiment"].astype(str), sent_df["n"],
    color=[sent_colors_map[s] for s in sent_df["sentiment"]],
    edgecolor="white", linewidth=0.5
)
for bar, val in zip(bars, sent_df["n"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
            f"{val:,}", ha="center", va="bottom", fontweight="bold", fontsize=11)
ax.set_title("Overall Sentiment Distribution Across All Treatment Reports",
             fontsize=13, fontweight="bold", pad=12)
ax.set_ylabel("Number of Reports")
ax.set_xlabel("Sentiment Category")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

**Interpretation:** The chart above shows how all 6,815 treatment reports break down by sentiment category. This baseline distribution reveals the overall patient experience landscape before drilling into individual treatments. A roughly even split between positive and negative sentiment (with mixed/neutral filling the middle) suggests that the community is candid about both successes and failures -- a healthy sign for data quality.

## 3. Top Treatments by Outcome (Binomial Test)

In [ ]:
# --- Load all reports joined with treatment names ---
reports = pd.read_sql("""
    SELECT tr.report_id, tr.user_id, tr.sentiment, tr.signal_strength,
           t.canonical_name AS treatment
    FROM treatment_reports tr
    JOIN treatment t ON tr.drug_id = t.id
""", conn)

reports["sent_score"] = reports["sentiment"].map(SENTIMENT_MAP)

# Filter out generic treatment names
reports = reports[~reports["treatment"].str.lower().isin(GENERICS)].copy()

display(HTML(
    f"<p>After filtering generics: <b>{len(reports):,}</b> reports across "
    f"<b>{reports['treatment'].nunique():,}</b> unique treatments from "
    f"<b>{reports['user_id'].nunique():,}</b> unique users.</p>"
))

# --- User-level aggregation ---
# For each user-treatment pair, take the mean sentiment score
user_level = (
    reports.groupby(["treatment", "user_id"])
    .agg(mean_sent=("sent_score", "mean"), n_reports=("report_id", "count"))
    .reset_index()
)

# Classify each user-treatment outcome
user_level["outcome"] = np.where(
    user_level["mean_sent"] > POS_THRESHOLD, "positive",
    np.where(user_level["mean_sent"] < NEG_THRESHOLD, "negative", "neutral/mixed")
)

# --- Aggregate per treatment ---
treat_agg = (
    user_level.groupby("treatment")
    .agg(
        n_users=("user_id", "nunique"),
        mean_sentiment=("mean_sent", "mean"),
        std_sentiment=("mean_sent", "std"),
        n_positive=("outcome", lambda x: (x == "positive").sum()),
        n_negative=("outcome", lambda x: (x == "negative").sum()),
        n_neutral=("outcome", lambda x: (x == "neutral/mixed").sum()),
    )
    .reset_index()
)

# Filter: minimum 10 unique users for statistical reliability
treat_agg = treat_agg[treat_agg["n_users"] >= 10].copy()

# --- Binomial test: is the positive rate significantly > 50%? ---
def binomial_p(row):
    """One-sided binomial test: H0: positive_rate <= 0.5."""
    successes = int(row["n_positive"])
    trials = int(row["n_users"])
    if trials == 0:
        return 1.0
    return stats.binomtest(successes, trials, p=0.5, alternative="greater").pvalue

treat_agg["p_value"] = treat_agg.apply(binomial_p, axis=1)
treat_agg["pos_rate"] = treat_agg["n_positive"] / treat_agg["n_users"]
treat_agg["neg_rate"] = treat_agg["n_negative"] / treat_agg["n_users"]
treat_agg["significant"] = treat_agg["p_value"] < 0.05
treat_agg = treat_agg.sort_values("pos_rate", ascending=False).reset_index(drop=True)

# --- Display top 20 ---
display_cols = ["treatment", "n_users", "n_positive", "n_negative", "n_neutral",
                "pos_rate", "neg_rate", "mean_sentiment", "p_value", "significant"]
top20 = treat_agg[display_cols].head(20).copy()
top20["pos_rate"] = top20["pos_rate"].map("{:.1%}".format)
top20["neg_rate"] = top20["neg_rate"].map("{:.1%}".format)
top20["mean_sentiment"] = top20["mean_sentiment"].map("{:.3f}".format)
top20["p_value"] = top20["p_value"].map("{:.4f}".format)

display(HTML("<h4>Top 20 Treatments by Positive Outcome Rate (min. 10 unique users, generics excluded)</h4>"))
display(top20)

display(HTML(
    "<p><b>Reading this table:</b> <code>pos_rate</code> is the share of unique users "
    "whose average sentiment score exceeded 0.7 (clearly positive). The <code>p_value</code> "
    "comes from a one-sided binomial test asking whether the positive rate is significantly "
    "greater than 50%. In plain language: a p-value below 0.05 means the treatment's "
    "positive rate is unlikely to be due to chance alone -- most users genuinely report "
    "good outcomes.</p>"
))

In [ ]:
# --- Diverging bar chart: top 15 treatments ---
# Rule: mixed/neutral INNERMOST (closest to center), negative OUTERMOST (furthest left)
top15 = treat_agg.head(15).copy().sort_values("pos_rate", ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))

labels = top15["treatment"].str.title()
y_pos = np.arange(len(labels))

n_pos = top15["n_positive"].values
n_neg = top15["n_negative"].values
n_neut = top15["n_neutral"].values
n_total = top15["n_users"].values

# Right side: positive counts
ax.barh(y_pos, n_pos, color=C_POS, label="Positive",
        edgecolor="white", linewidth=0.5)

# Left side: neutral/mixed INNERMOST (closest to center), then negative OUTERMOST
ax.barh(y_pos, -n_neut, color=C_NEUT, label="Neutral/Mixed",
        edgecolor="white", linewidth=0.5)
ax.barh(y_pos, -n_neg, left=-n_neut, color=C_NEG, label="Negative",
        edgecolor="white", linewidth=0.5)

ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel("Number of Users")
ax.set_title("Top 15 Treatments: Diverging Outcome Bar Chart\n(Positive right, Negative/Mixed left)",
             fontsize=13, fontweight="bold")
ax.axvline(0, color="black", linewidth=0.8)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=3, frameon=False)

# Annotate total users on the right end
for i, total in enumerate(n_total):
    ax.text(n_pos[i] + 1.5, i, f"n={total}", va="center", fontsize=8, color="#555")

plt.tight_layout()
plt.show()

**Interpretation:** The diverging bar chart shows the distribution of user outcomes for each treatment. Bars extending to the right represent users with clearly positive outcomes (mean sentiment > 0.7); bars extending to the left show neutral/mixed outcomes (inner, gray) and negative outcomes (outer, red). Treatments are sorted by positive outcome rate from bottom to top. Treatments with bars heavily skewed right, combined with sufficient sample sizes (shown as n= annotations), are the strongest candidates for further investigation. In plain language: the further right the green bar extends relative to the red, the more consistently users report benefit.

## 4. Precision: Mean Sentiment with 95% Confidence Intervals

In [ ]:
# --- Forest plot: mean sentiment +/- 95% CI for top 15 ---
top15_forest = treat_agg.head(15).copy().sort_values("mean_sentiment", ascending=True)

ci_data = []
for _, row in top15_forest.iterrows():
    t_name = row["treatment"]
    user_means = user_level[user_level["treatment"] == t_name]["mean_sent"]
    n = len(user_means)
    mean_val = user_means.mean()
    se = user_means.std(ddof=1) / np.sqrt(n) if n > 1 else 0
    ci_low = mean_val - 1.96 * se
    ci_high = mean_val + 1.96 * se
    ci_data.append({"treatment": t_name, "mean": mean_val,
                    "ci_low": ci_low, "ci_high": ci_high, "n": n, "se": se})

ci_df = pd.DataFrame(ci_data)

fig, ax = plt.subplots(figsize=(10, 8))
y_pos = np.arange(len(ci_df))

# Color by whether CI excludes 0
colors = [C_POS if row["ci_low"] > 0 else (C_NEG if row["ci_high"] < 0 else C_NEUT)
          for _, row in ci_df.iterrows()]

ax.errorbar(ci_df["mean"], y_pos,
            xerr=[ci_df["mean"] - ci_df["ci_low"], ci_df["ci_high"] - ci_df["mean"]],
            fmt="none", ecolor="#555", elinewidth=1.5, capsize=4)
ax.scatter(ci_df["mean"], y_pos, c=colors, s=80, zorder=5,
           edgecolors="white", linewidth=0.5)

ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_yticks(y_pos)
ax.set_yticklabels(ci_df["treatment"].str.title(), fontsize=10)
ax.set_xlabel("Mean Sentiment Score (user-level)")
ax.set_title("Forest Plot: Mean Sentiment with 95% CI (Top 15 Treatments)",
             fontsize=13, fontweight="bold")

for i in range(len(ci_df)):
    ax.text(ci_df.iloc[i]["ci_high"] + 0.03, i,
            f"n={ci_df.iloc[i]['n']}", va="center", fontsize=8, color="#555")

legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=C_POS, markersize=10,
           label="CI entirely above 0 (positive signal)"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor=C_NEUT, markersize=10,
           label="CI spans 0 (uncertain)"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor=C_NEG, markersize=10,
           label="CI entirely below 0 (negative signal)"),
]
ax.legend(handles=legend_elements, loc="upper center", bbox_to_anchor=(0.5, -0.06),
          ncol=3, frameon=False, fontsize=9)

plt.tight_layout()
plt.show()

**Interpretation:** The forest plot shows the mean user-level sentiment score and its 95% confidence interval for each treatment. Green dots indicate treatments whose confidence interval lies entirely above zero, meaning we can be fairly confident the average user experience is positive. Gray dots indicate treatments whose CI crosses zero, suggesting uncertain or inconsistent outcomes. The width of each CI reflects both sample size and consistency: narrower intervals signal more precise (and therefore more reliable) estimates. In plain language: if the green dot and its error bar are entirely to the right of the dashed line, the treatment is working for most people who tried it.

## 5. Condition Prevalence

In [ ]:
# --- Condition prevalence from the conditions table ---
conditions_df = pd.read_sql("""
    SELECT condition_name, COUNT(DISTINCT user_id) AS n_users
    FROM conditions
    GROUP BY condition_name
    ORDER BY n_users DESC
""", conn)

if len(conditions_df) == 0:
    display(HTML(
        "<p><i>No condition data available in this dataset. "
        "Condition extraction may not have been run on this database.</i></p>"
    ))
else:
    top_cond = conditions_df.head(20)
    display(HTML(f"<h4>Top {len(top_cond)} Conditions by Unique Users</h4>"))
    display(top_cond)

    # Bar chart of top 15 conditions
    plot_cond = conditions_df.head(15).sort_values("n_users", ascending=True)

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(
        plot_cond["condition_name"].str.title(),
        plot_cond["n_users"],
        color="#3498db", edgecolor="white", linewidth=0.5
    )
    for bar, val in zip(bars, plot_cond["n_users"]):
        ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                f"{val}", va="center", fontsize=9, color="#333")
    ax.set_xlabel("Number of Unique Users")
    ax.set_title("Top 15 Conditions Reported by Users in the Dataset",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

    display(HTML(
        "<p><b>Interpretation:</b> This chart shows the most commonly reported conditions "
        "among users in the dataset. The prevalence of comorbid conditions like POTS, MCAS, "
        "dysautonomia, and ME/CFS alongside Long COVID reflects the multi-system nature of "
        "the illness and influences which treatments users gravitate toward. For example, "
        "antihistamine use correlates with MCAS prevalence, and beta-blocker use with POTS.</p>"
    ))

## 6. Tiered Recommendations

In [ ]:
# --- Tier classification ---
# Strong:      p < 0.01  AND  n_users >= 20  AND  pos_rate >= 0.50
# Moderate:    p < 0.05  AND  n_users >= 15  AND  pos_rate >= 0.40
# Preliminary: p < 0.10  AND  n_users >= 10  AND  pos_rate >= 0.35

def assign_tier(row):
    if row["p_value"] < 0.01 and row["n_users"] >= 20 and row["pos_rate"] >= 0.50:
        return "Strong"
    elif row["p_value"] < 0.05 and row["n_users"] >= 15 and row["pos_rate"] >= 0.40:
        return "Moderate"
    elif row["p_value"] < 0.10 and row["n_users"] >= 10 and row["pos_rate"] >= 0.35:
        return "Preliminary"
    return None

treat_agg["tier"] = treat_agg.apply(assign_tier, axis=1)
tiered = treat_agg[treat_agg["tier"].notna()].copy()
tiered["tier"] = pd.Categorical(
    tiered["tier"], categories=["Strong", "Moderate", "Preliminary"], ordered=True
)
tiered = tiered.sort_values(["tier", "pos_rate"], ascending=[True, False])

# Display criteria explanation
display(HTML(
    "<h4>Tiered Treatment Recommendations</h4>"
    "<ul>"
    "<li><b style='color:#27ae60'>Strong:</b> p &lt; 0.01, n &ge; 20 users, positive rate &ge; 50%. "
    "In plain language: a large number of users tried this, and significantly more than half report "
    "clearly positive outcomes. The statistical test confirms this is very unlikely to be chance.</li>"
    "<li><b style='color:#f39c12'>Moderate:</b> p &lt; 0.05, n &ge; 15 users, positive rate &ge; 40%. "
    "A promising signal from a decent sample. The positive rate is statistically significant at the "
    "conventional threshold, but confidence is somewhat lower.</li>"
    "<li><b style='color:#3498db'>Preliminary:</b> p &lt; 0.10, n &ge; 10 users, positive rate &ge; 35%. "
    "An early signal worth monitoring as more data accumulates. The evidence is suggestive but not "
    "yet strong enough for high confidence.</li>"
    "</ul>"
))

# Display tiered table
tier_display = tiered[["tier", "treatment", "n_users", "pos_rate", "neg_rate",
                        "mean_sentiment", "p_value"]].copy()
tier_display["pos_rate"] = tier_display["pos_rate"].map("{:.1%}".format)
tier_display["neg_rate"] = tier_display["neg_rate"].map("{:.1%}".format)
tier_display["mean_sentiment"] = tier_display["mean_sentiment"].map("{:.3f}".format)
tier_display["p_value"] = tier_display["p_value"].map("{:.4f}".format)
display(tier_display)

In [ ]:
# --- Tiered forest plot color-coded by evidence tier ---
if len(tiered) == 0:
    display(HTML("<p><i>No treatments met the tiering criteria.</i></p>"))
else:
    tiered_plot = tiered.sort_values("mean_sentiment", ascending=True).copy()

    tier_ci = []
    for _, row in tiered_plot.iterrows():
        t_name = row["treatment"]
        user_means = user_level[user_level["treatment"] == t_name]["mean_sent"]
        n = len(user_means)
        mean_val = user_means.mean()
        se = user_means.std(ddof=1) / np.sqrt(n) if n > 1 else 0
        tier_ci.append({"treatment": t_name, "mean": mean_val,
                        "ci_low": mean_val - 1.96 * se,
                        "ci_high": mean_val + 1.96 * se,
                        "n": n, "tier": row["tier"]})
    tier_ci_df = pd.DataFrame(tier_ci)

    fig_height = max(5, len(tier_ci_df) * 0.5)
    fig, ax = plt.subplots(figsize=(11, fig_height))
    y_pos = np.arange(len(tier_ci_df))

    colors = [TIER_COLORS.get(str(t), C_NEUT) for t in tier_ci_df["tier"]]

    ax.errorbar(tier_ci_df["mean"], y_pos,
                xerr=[tier_ci_df["mean"] - tier_ci_df["ci_low"],
                      tier_ci_df["ci_high"] - tier_ci_df["mean"]],
                fmt="none", ecolor="#555", elinewidth=1.5, capsize=4)
    ax.scatter(tier_ci_df["mean"], y_pos, c=colors, s=100, zorder=5,
               edgecolors="white", linewidth=0.5)

    ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(tier_ci_df["treatment"].str.title(), fontsize=10)
    ax.set_xlabel("Mean Sentiment Score (user-level)")
    ax.set_title("Tiered Recommendations: Forest Plot with 95% Confidence Intervals",
                 fontsize=13, fontweight="bold")

    for i in range(len(tier_ci_df)):
        ax.text(tier_ci_df.iloc[i]["ci_high"] + 0.03, i,
                f"n={tier_ci_df.iloc[i]['n']}", va="center", fontsize=8, color="#555")

    legend_elements = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=TIER_COLORS["Strong"],
               markersize=10, label="Strong evidence"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor=TIER_COLORS["Moderate"],
               markersize=10, label="Moderate evidence"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor=TIER_COLORS["Preliminary"],
               markersize=10, label="Preliminary evidence"),
    ]
    ax.legend(handles=legend_elements, loc="upper center", bbox_to_anchor=(0.5, -0.06),
              ncol=3, frameon=False, fontsize=9)

    plt.tight_layout()
    plt.show()

**Interpretation:** This forest plot color-codes each treatment by its evidence tier. Green (Strong) treatments have the most robust evidence: large sample sizes, high positive rates, and highly significant p-values. Orange (Moderate) treatments show promising signals but with wider uncertainty. Blue (Preliminary) treatments represent early signals from smaller samples that warrant continued monitoring. In plain language: green dots are the best bets from this data, orange are worth considering, and blue are ones to watch.

## 7. Summary and Disclaimer

In [ ]:
# --- Final summary ---
n_strong = len(tiered[tiered["tier"] == "Strong"]) if len(tiered) > 0 else 0
n_moderate = len(tiered[tiered["tier"] == "Moderate"]) if len(tiered) > 0 else 0
n_prelim = len(tiered[tiered["tier"] == "Preliminary"]) if len(tiered) > 0 else 0

strong_list = tiered[tiered["tier"] == "Strong"]["treatment"].str.title().tolist()[:5]
strong_str = ", ".join(strong_list) if strong_list else "(none met criteria)"
mod_list = tiered[tiered["tier"] == "Moderate"]["treatment"].str.title().tolist()[:5]
mod_str = ", ".join(mod_list) if mod_list else "(none met criteria)"

display(HTML(f"""
<h4>Summary</h4>
<p>Across <b>{counts['unique_reporters']:,}</b> unique users and 
<b>{counts['treatment_reports']:,}</b> treatment reports from 
{d_min.strftime('%B %Y')} to {d_max.strftime('%B %Y')}, this analysis identified:</p>
<ul>
    <li><b>{n_strong}</b> treatments with 
        <span style="color:{TIER_COLORS['Strong']};font-weight:bold">Strong</span> 
        evidence: {strong_str}</li>
    <li><b>{n_moderate}</b> treatments with 
        <span style="color:{TIER_COLORS['Moderate']};font-weight:bold">Moderate</span> 
        evidence: {mod_str}</li>
    <li><b>{n_prelim}</b> treatments with 
        <span style="color:{TIER_COLORS['Preliminary']};font-weight:bold">Preliminary</span> 
        evidence</li>
</ul>
<p>User-level aggregation ensures prolific posters do not inflate any single treatment's 
apparent success rate. Binomial testing and confidence intervals provide a statistical 
framework for separating signal from noise.</p>

<h4>Disclaimer</h4>
<div style="background:#fff3cd;padding:15px;border-radius:8px;border-left:4px solid #ffc107;margin:10px 0">
<p><b>Reporting Bias Warning:</b> This analysis is based entirely on self-reported patient 
experiences from Reddit. The data is subject to multiple biases:</p>
<ul>
    <li><b>Selection bias:</b> Users who post are not representative of all Long COVID patients</li>
    <li><b>Positive reporting bias:</b> People may be more motivated to share positive outcomes</li>
    <li><b>Survivorship bias:</b> Users who discontinue ineffective treatments may stop posting</li>
    <li><b>Placebo effect:</b> Self-reported improvements may not reflect pharmacological efficacy</li>
    <li><b>Confounding:</b> Users typically try multiple treatments simultaneously, making it 
        impossible to attribute improvement to any single intervention</li>
    <li><b>Temporal bias:</b> This dataset covers approximately one month and may not capture 
        long-term outcomes</li>
</ul>
<p>These findings are <b>hypothesis-generating only</b> and should not be used to make 
clinical decisions. Always consult a qualified healthcare provider before starting, stopping, 
or changing any treatment.</p>
</div>
"""))

conn.close()
display(HTML("<p><i>Analysis complete. Database connection closed.</i></p>"))